# **Day 52: Enums**

An **enum** (enumeration) is a fixed set of **named constant values**.

**Why use enums?** If we use plain text like `"pending"`, someone may type `"Pending "` or `"pendng"` by mistake, and Python will not notice. With an enum, a typing mistake gives an error **immediately**.

In [1]:
from enum import Enum, auto

class OrderStatus(Enum):
    PLACED = 1
    PACKED = 2
    SHIPPED = 3
    DELIVERED = 4
    CANCELLED = 5

s = OrderStatus.SHIPPED
print(s, s.name, s.value)
print(OrderStatus(4))              # Get a member by its value
print(OrderStatus["PACKED"])       # Get a member by its name

# OrderStatus.SHIPED               # This will raise an AttributeError (spelling mistake)

OrderStatus.SHIPPED SHIPPED 3
OrderStatus.DELIVERED
OrderStatus.PACKED


In [2]:
# Loop over all members
for st in OrderStatus:
    print(st.value, st.name)

1 PLACED
2 PACKED
3 SHIPPED
4 DELIVERED
5 CANCELLED


## **Example from Remote Sensing: Land Cover Classes**
A **classified satellite image** stores a **number** in every pixel, and each number means a land cover class. An enum is the perfect way to keep these codes in one place.

We can also add **methods** to an enum, for example to give each class a map colour.

In [3]:
class LandCover(Enum):
    WATER = 1
    FOREST = 2
    AGRICULTURE = 3
    BUILT_UP = 4
    BARE_LAND = 5
    MANGROVE = 6

    def colour(self):
        return {
            LandCover.WATER: "blue",
            LandCover.FOREST: "dark green",
            LandCover.AGRICULTURE: "yellow",
            LandCover.BUILT_UP: "red",
            LandCover.BARE_LAND: "brown",
            LandCover.MANGROVE: "teal",
        }[self]

    def is_vegetation(self):
        return self in (LandCover.FOREST, LandCover.AGRICULTURE, LandCover.MANGROVE)

for lc in LandCover:
    print(f"{lc.value}  {lc.name:<12} colour: {lc.colour():<11} vegetation: {lc.is_vegetation()}")

1  WATER        colour: blue        vegetation: False
2  FOREST       colour: dark green  vegetation: True
3  AGRICULTURE  colour: yellow      vegetation: True
4  BUILT_UP     colour: red         vegetation: False
5  BARE_LAND    colour: brown       vegetation: False
6  MANGROVE     colour: teal        vegetation: True


In [4]:
# A tiny classified image (each number is a pixel's class code)
classified = [
    [1, 1, 6, 6, 3],
    [1, 6, 6, 3, 3],
    [6, 6, 4, 4, 3],
    [2, 2, 4, 5, 3],
]

counts = {}
for row in classified:
    for code in row:
        lc = LandCover(code)                  # Turn the number into an enum member
        counts[lc] = counts.get(lc, 0) + 1

total = sum(counts.values())
for lc, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"{lc.name:<12}{n:>3} pixels  {n / total:6.1%}")

veg = sum(n for lc, n in counts.items() if lc.is_vegetation())
print(f"Total vegetation cover: {veg / total:.0%}")

MANGROVE      6 pixels   30.0%
AGRICULTURE   5 pixels   25.0%
WATER         3 pixels   15.0%
BUILT_UP      3 pixels   15.0%
FOREST        2 pixels   10.0%
BARE_LAND     1 pixels    5.0%
Total vegetation cover: 65%


Let us understand the code above:
- `LandCover(code)` changes a number like `6` into `LandCover.MANGROVE`.
- Now the code talks about "MANGROVE" instead of a meaningless number 6, so it is much easier to read.
- `is_vegetation()` groups forest, agriculture and mangrove together.

## **Comparing Enum Members**

In [5]:
s = OrderStatus.DELIVERED
print(s is OrderStatus.DELIVERED)
print(s == 4)          # False: an enum member is not equal to its plain value

True
False


## **auto(): Automatic Values**

In [6]:
class Division(Enum):
    PRESIDENCY = auto()
    BURDWAN = auto()
    MEDINIPUR = auto()
    MALDA = auto()
    JALPAIGURI = auto()

print([(d.name, d.value) for d in Division])

[('PRESIDENCY', 1), ('BURDWAN', 2), ('MEDINIPUR', 3), ('MALDA', 4), ('JALPAIGURI', 5)]


## **IntEnum and StrEnum**
These enums **behave like** normal numbers or text, which is useful when saving to files or databases.

In [7]:
from enum import IntEnum, StrEnum

class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3

print(Priority.HIGH > Priority.LOW, Priority.HIGH + 1)

class Satellite(StrEnum):
    LANDSAT8 = "Landsat 8"
    SENTINEL2 = "Sentinel-2"

print(Satellite.SENTINEL2.upper(), Satellite.SENTINEL2 == "Sentinel-2")

True 4
SENTINEL-2 True


## **Practice Problem: Allowed Status Changes**
An order can only move in certain ways. For example, a delivered order cannot go back to "packed".

In [8]:
ALLOWED = {
    OrderStatus.PLACED: {OrderStatus.PACKED, OrderStatus.CANCELLED},
    OrderStatus.PACKED: {OrderStatus.SHIPPED, OrderStatus.CANCELLED},
    OrderStatus.SHIPPED: {OrderStatus.DELIVERED},
    OrderStatus.DELIVERED: set(),
    OrderStatus.CANCELLED: set(),
}

class Order:
    def __init__(self, oid):
        self.oid = oid
        self.status = OrderStatus.PLACED

    def move(self, new):
        if new not in ALLOWED[self.status]:
            print(f"  Order {self.oid}: cannot go from {self.status.name} to {new.name}")
            return
        print(f"  Order {self.oid}: {self.status.name} -> {new.name}")
        self.status = new

o = Order(501)
o.move(OrderStatus.PACKED)
o.move(OrderStatus.DELIVERED)
o.move(OrderStatus.SHIPPED)
o.move(OrderStatus.DELIVERED)
o.move(OrderStatus.CANCELLED)

  Order 501: PLACED -> PACKED
  Order 501: cannot go from PACKED to DELIVERED
  Order 501: PACKED -> SHIPPED
  Order 501: SHIPPED -> DELIVERED
  Order 501: cannot go from DELIVERED to CANCELLED


In [9]:
# Quick question
# Guess the output of this code cell
# from enum import Enum
# class Colour(Enum):
#     RED = 1
#     GREEN = 2
# print(Colour(2).name, Colour.RED.value)